# Day 16 · Collaborative map — ใครเขียน OpenStreetMap รอบฟ้ามืดของไทย

**#30DayMapChallenge 2026 · 16 พ.ย. 2569** (วัดและเขียนเช้าวันที่ 17 — ดูเหตุผลในโพสต์) ·
โจทย์: *"Do a map with a friend, colleague or a stranger. AI doesn't count as a friend :)"*

ผมชวนมนุษย์ไม่ทัน และ AI ไม่นับเป็นผู้ร่วมงานตามที่เขาเขียนกำกับไว้
ทางที่เหลือคือวัดว่า *ใครคือคนที่ทำงานนี้ด้วยจริง ๆ* — คำตอบคือคนแปลกหน้า 299 คน
ที่เขียน OpenStreetMap รอบจุดทะเบียนฟ้ามืด 83 จุด

| วัด | นิยาม |
|---|---|
| ปริมาณแผนที่ | จำนวน `way` ที่จุดศูนย์กลางอยู่ภายใน 1 กม. จากพิกัดจุด (ไม่นับ node) |
| จำนวนคน | username ที่ต่างกันของ *การแก้ไขล่าสุด* ของ way เหล่านั้น |
| การพึ่งพาคนเดียว | ส่วนแบ่ง way ของคนที่อันดับหนึ่ง (ร้อยละ) |
| ความเป็นชุมชน | จำนวนคนที่ปรากฏในกล่องของมากกว่าหนึ่งจุดทะเบียน |
| เทียบกับความมืด | ประชากรเมืองรอบจุดจาก Day 9 + ผู้ถือครองจากฟิลด์ type ของทะเบียน |

**รัน 2 ไฟล์ตามลำดับ**
1. `tools/collect_day16_osm.py` → `cache/day16/way_all.json` (มี resume, ยิงใหม่เฉพาะชุดที่ขาด)
   และ `--compare` เพื่อเทียบนิยาม "แตะกล่อง" กับ "จุดศูนย์กลางในกล่อง"
2. `tools/day16_collaborative.py` → ภาพ 3 แผ่น + `day16_sites_osm.csv` / `day16_by_holder.csv` /
   `day16_stats.json`

**ข้อควรทราบก่อนรัน**
- query ต้องจบด้วย `out meta;` ไม่ใช่ `out;` ไม่งั้นคอลัมน์ user/timestamp จะว่างเปล่าเงียบ ๆ
- Overpass คืนผลลัพธ์เป็น **TSV** (tab) ไม่ใช่ CSV ที่มีจุลภาค
- `changesets?bbox=` ใช้กับคำถามนี้ไม่ได้ bbox ของ changeset ครอบคลุมทั้งประเทศ
  (จุดห่าง 1,463 กม. แชร์ changeset 44 จาก 100 ชุด) — ลองแล้วทิ้ง ดู `failed_attempts` ใน stats
- ข้อมูลแผนที่ใช้ภายใต้ © OpenStreetMap contributors (ODbL)

### ค่าคงที่ + ชุดเครื่องมือ (อ่าน/ดาวน์โหลดข้อมูล, ตั้งฟอนต์ไทย, ตรวจว่ารันที่ไหน)

## ขั้นที่ 0 — เก็บข้อมูลก่อน (ต้องรันบนเครื่องที่มีเน็ต)

เซลล์ถัดไปคือ `tools/collect_day16_osm.py` ต้นทาง ต้องรัน *ก่อน*
ให้ได้ออกมาเป็น `cache/day16/way_all.json` (มี resume — ข้ามชุดที่ครบแล้ว)

```bash
python tools/collect_day16_osm.py            # 83 จุด
python tools/collect_day16_osm.py --compare   # เทียบสองนิยามที่ 3 จุด
```

In [ ]:
# -*- coding: utf-8 -*-
"""
collect_day16_osm.py — ดึง "ใครเขียนแผนที่รอบจุดทะเบียนฟ้ามืด" จาก OpenStreetMap (Overpass)

โหมดรวมชุด (batched) เพราะ Overpass คิวแน่นมาก: ยิงทีละจุด 83 ครั้งไม่จบในเช้าวันนี้
จึงรวมหลายจุดใน query เดียวด้วย `out center meta;` ซึ่งคืน id, user, timestamp
และ *จุดศูนย์กลางของ way* → จับคู่ way กับจุดทะเบียนด้วยระยะทางจากจุดศูนย์กลางนั้น

นิยามที่โพสต์ใช้ (ต้องตรงกับโค้ดนี้):
    way ที่จุดศูนย์กลางของ way อยู่ภายใน 1 กม. จากพิกัดจุดทะเบียน
  (ต่างจาก `around:1000` ที่นับ way ที่ส่วนใดส่วนหนึ่งของวัตถุแตะกล่อง —
   วัดเทียบไว้ที่หลาจุดใน cache/day16/scope_compare.json ผ่าน `--compare`)

รัน:  python tools/collect_day16_osm.py            # ทั้ง 83 จุด
     python tools/collect_day16_osm.py --compare   # เทียบสองนิยามที่ 3 จุด
"""
import os, sys, json, time, math, urllib.parse, urllib.request
import pandas as pd

WORK = r"D:\GIS30Days"
REPO = os.path.join(WORK, "geo-notes")
CACHE = os.path.join(WORK, "cache", "day16")
os.makedirs(CACHE, exist_ok=True)
ENDPOINTS = ["https://overpass-api.de/api/interpreter",
             "https://maps.mail.ru/osm/tools/overpass/api/interpreter",
             "https://overpass.kumi.systems/api/interpreter",
             "https://overpass.private.coffee/api/interpreter"]
RADIUS_M = 1000
BATCH = 3
UA = "geo-notes-day16/1.0 (30DayMapChallenge research; one-off study)"
OUT_PATH = os.path.join(CACHE, "way_all.json")


def log(m):
    print(f"[{time.strftime('%H:%M:%S')}] {m}", flush=True)


def query_batch(sites):
    lines = ["[out:csv(::id,::user,::timestamp,::lat,::lon,false)][timeout:300];", "("]
    for lat, lon in sites:
        lines.append(f"  way(around:{RADIUS_M},{lat:.6f},{lon:.6f});")
    lines += [");", "out center meta;"]
    return chr(10).join(lines) + chr(10)


def fetch(q, rounds=10):
    body = urllib.parse.urlencode({"data": q}).encode()
    for a in range(rounds):
        for ep in ENDPOINTS:
            try:
                req = urllib.request.Request(ep, data=body, headers={
                    "User-Agent": UA, "Content-Type": "application/x-www-form-urlencoded"})
                with urllib.request.urlopen(req, timeout=90) as r:
                    txt = r.read().decode("utf-8", "replace")
            except Exception as e:
                log(f"  {ep.split('/')[2]} -> {type(e).__name__} {str(e)[:60]}")
                time.sleep(4 + 3 * a)
                continue
            if "Gateway" in txt[:200] or txt.count(chr(10)) < 2:
                log(f"  {ep.split('/')[2]} -> busy ({txt[:60]!r})")
                time.sleep(4 + 3 * a)
                continue
            return txt
    return None


def hav_m(a, b):
    la1, lo1 = math.radians(a[0]), math.radians(a[1])
    la2, lo2 = math.radians(b[0]), math.radians(b[1])
    d = math.sin((la2 - la1) / 2) ** 2 + math.cos(la1) * math.cos(la2) * math.sin((lo2 - lo1) / 2) ** 2
    return 2 * 6371008.8 * math.asin(math.sqrt(d))


def parse(txt, sites):
    """แบ่ง way ให้จุดทะเบียนด้วยจุดศูนย์กลางของ way (กัน way ยาว ๆ ที่ลากผ่านหลายกล่อง)"""
    rows = [l.split(chr(9)) for l in txt.splitlines()[1:] if l.count(chr(9)) >= 4]
    out = {i: [] for i in range(len(sites))}
    far = 0
    for r in rows:
        uid, ts, lat, lon = r[1].strip('"'), r[2].strip('"'), float(r[3]), float(r[4])
        if not uid:
            continue
        dmin, i = min((hav_m((lat, lon), (s[0], s[1])), j) for j, s in enumerate(sites))
        if dmin <= RADIUS_M:
            out[i].append([r[0].strip('"'), uid, ts, lat, lon])
        else:
            far += 1
    return out, far


df = pd.read_csv(os.path.join(REPO, "data", "darksky_thailand.csv"))
df["name"] = df["name"].astype(str).str.strip()
SITES = [(float(r.latitude), float(r.longitude)) for _, r in df.iterrows()]

if "--compare" in sys.argv:
    res = {}
    for i in (0, 30, 62):
        lat, lon = SITES[i]
        q = (f"[out:csv(::id,::user,::timestamp,false)][timeout:200];\n"
             f"way(around:{RADIUS_M},{lat:.6f},{lon:.6f});\nout meta;\n")
        txt = fetch(q)
        if txt is None:
            log(f"จุด {i} เทียบไม่ได้ (server busy)")
            continue
        body = [l.split(chr(9)) for l in txt.splitlines()[1:] if l.count(chr(9)) >= 2]
        txt2 = fetch(query_batch([SITES[i]]))
        got, _ = parse(txt2, [SITES[i]]) if txt2 else ({0: []}, 0)
        res[str(i)] = dict(name=df.name.iloc[i], around_ways=len(body),
                           around_users=len({x[1] for x in body}),
                           center_ways=len(got[0]), center_users=len({x[1] for x in got[0]}))
        log(f"จุด {i} {df.name.iloc[i][:24]}: around {res[str(i)]['around_ways']} way/"
            f"{res[str(i)]['around_users']} คน vs center {res[str(i)]['center_ways']} way/"
            f"{res[str(i)]['center_users']} คน")
        time.sleep(4)
    json.dump(res, open(os.path.join(CACHE, "scope_compare.json"), "w", encoding="utf-8"),
              ensure_ascii=False, indent=1)
    print("COMPARE_DONE")
    sys.exit()

# resume: อ่านผลชุดที่ทำเสร็จแล้วกลับมา ไม่ยิงซ้ำ
results, far_total = {}, 0
if os.path.exists(OUT_PATH):
    prev = json.load(open(OUT_PATH, encoding="utf-8"))
    results = {k: v for k, v in prev.get("sites", {}).items()}
    far_total = int(prev.get("dropped_far_ways_total", 0))
    log(f"อ่านแคชเดิม: ได้ {len(results)} จุดแล้ว จะข้ามชุดที่ครบ")
done_batches = {int(k) // BATCH for k in results}
for b in range(0, len(SITES), BATCH):
    if all(str(i) in results for i in range(b, min(b + BATCH, len(SITES)))):
        continue
    chunk = SITES[b:b + BATCH]
    log(f"ชุดที่ {b//BATCH + 1}: จุด {b}–{b + len(chunk) - 1}")
    txt = fetch(query_batch(chunk))
    if txt is None:
        raise SystemExit(f"ชุด {b} พังทุก endpoint — รันใหม่ (ผลชุดก่อนหน้าถูกบันทึกไว้แล้ว)")
    got, far = parse(txt, chunk)
    far_total += far
    for j, rows in got.items():
        results[str(b + j)] = rows
    json.dump({"radius_m": RADIUS_M, "attribution": "จุดศูนย์กลางของ way อยู่ภายใน 1 กม. จากจุด",
               "fetched_utc": time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime()),
               "dropped_far_ways_total": far_total,
               "sites": {str(k): v for k, v in results.items()}},
              open(OUT_PATH, "w", encoding="utf-8"), ensure_ascii=False)
    tot = sum(len(v) for v in results.values())
    log(f"  สะสม way {tot:,} ชิ้น · ทิ้งเพราะจุดศูนย์กลางไกลเกิน 1 กม. {far_total:,}")
    time.sleep(5)

have = len(results)
log(f"ครบ {have} / {len(SITES)} จุด · way รวม {sum(len(v) for v in results.values()):,}")
print("COLLECT_DONE", have)

In [ ]:
# -*- coding: utf-8 -*-
"""
Day 16 · Collaborative map — "Do a map with a friend, colleague or a stranger.
AI doesn't count as a friend :)"

ข้อความโจทย์ดึงจาก 30daymapchallenge.com อ่านสด (07:00 ของวันที่ 17 พ.ย.)

เมื่อคืนผมชวนใครไม่ทัน และผมไม่สมมติว่า AI เป็นเพื่อน ทางที่เหลือคือทำแผนที่ที่
*เป็น* collaborative อยู่จริงและพิสูจน์ได้: วัดว่าใครเป็นคนเขียน OpenStreetMap
รอบจุดทะเบียนฟ้ามืด 83 จุดของไทย แล้วพบว่าคำตอบไม่ได้บอกว่า "แผนที่เขียนโดยคนมากมาย"
แต่บอกว่า *ที่ที่ฟ้าดีที่สุด คือที่ที่แผนที่บางที่สุด*

นิยามที่วัด (ต้องตรงกับ collect_day16_osm.py):
    way ที่จุดศูนย์กลางของ way อยู่ภายใน 1 กม. จากพิกัดจุดทะเบียน · นับเฉพาะ way ไม่ นับ node

บันทึกความล้มเหลวที่กลายเป็นส่วนหนึ่งของงาน (โพสต์อ้างถึง):
  • คืนวันที่ 16 overpass-api.de ไม่ตอบ / mirror คืนคอลัมน์ user ว่างเปล่าเพราะ query จบด้วย
    `out;` แทนที่จะเป็น `out meta;`
  • ลองวัดด้วย changesets API แล้วทิ้ง — bbox ของ changeset ครอบคลุมทั้งประเทศ
    จุดใต้สุดกับจุดเหนือสุดห่าง 1,463 กม. แต่ยังแชร์ changeset กัน 44 จาก 100 ชุด
    (ไฟล์ครั้งนั้นตั้งชื่อ INVALID_ ไว้ใน out/day16 ไม่ถูกใช้ที่ไหนอีก)

ทุกตัวเลขอ่านจาก day16_sites_osm.csv / day16_by_holder.csv / day16_stats.json เท่านั้น
"""
import os, sys, json, time, collections
import numpy as np, pandas as pd, geopandas as gpd
import h3
from shapely.validation import make_valid
from pyproj import CRS
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.collections import PolyCollection
from scipy.stats import spearmanr

T0 = time.time()
WORK = r"D:\GIS30Days"
REPO = os.path.join(WORK, "geo-notes")
CACHE = os.path.join(WORK, "cache", "day06")
OSMC = os.path.join(WORK, "cache", "day16")
OUT = os.path.join(WORK, "out", "day16")
os.makedirs(OUT, exist_ok=True)
for _f in sorted(os.listdir(CACHE)):
    if _f.endswith(".ttf"):
        font_manager.fontManager.addfont(os.path.join(CACHE, _f))
plt.rcParams["font.family"] = "Trirong"
_LOGF = open(os.path.join(OUT, "day16_run.log"), "w", encoding="utf-8")
def log(m):
    s = f"[{time.time()-T0:6.1f}s] {m}"
    print(s, flush=True)
    _LOGF.write(s + chr(10)); _LOGF.flush()

sys.path.insert(0, os.path.join(WORK, "tools"))
from guards import guard_bbox, guard_crs, guard_variance, lint_thai, hav_km

TXT = []
def T(s):
    TXT.append(s); return s

EQ = CRS.from_proj4("+proj=eqdc +lat_0=14 +lon_0=101 +lat_1=8 +lat_2=20 +units=m +no_defs +ellps=WGS84")
BG, INK, MUTED = "#0d1522", "#e8e3d3", "#9fb2c8"
ACCENT, WARM, COOL, TEAL = "#f5b942", "#f4566a", "#38bdf8", "#2dd4bf"
ATTRIBUTION = "© OpenStreetMap contributors (ODbL)"

## ตอน 1) ข้อมูลตั้งต้นของซีรีส์

In [ ]:
# ---------------------------------------------------------------- 1. ข้อมูลตั้งต้นของซีรีส์
df = pd.read_csv(os.path.join(REPO, "data/darksky_thailand.csv"))
df["name"] = df["name"].astype(str).str.strip()
assert len(df) == 83, f"ทะเบียนต้องมี 83 จุด ได้ {len(df)}"
SITES = np.column_stack([df.longitude.values, df.latitude.values])
guard_bbox(SITES, 96.5, 107.6, 4.3, 21.7, "darksky-sites")

g3 = gpd.read_file(os.path.join(REPO, "posts/day03-polygons/day03_hex_polygons.geojson"))
cell_ids = np.array([str(c) for c in g3.cell.values])
cen_deg = np.array([[c.centroid.x, c.centroid.y] for c in g3.geometry.values])
d9 = gpd.read_file(os.path.join(REPO, "posts/day09-urban-rural/day09_hex_classes.geojson"))
assert len(d9) == len(g3), "day09 กับ day03 ต้องเรียง hex ตรงกัน (ตรวจแล้วตั้งแต่ Day 13)"
urban_pop = np.array([float(x) for x in d9.urban_pop_30km.values])
# ผู้ถือครอง = ฟิลด์ type ของทะเบียนเอง (ย่อตามแผนที่ Day 12 ใช้) — ไม่ join ผ่าน hex
# เพราะ day12_hex_holders.geojson ไม่มีฟิลด์ cell ให้จับคู่ (ตรวจแล้ว) การยึด hex จึงได้ "(ไม่ระบุ)"
SHORT = {"เขตอนุรักษ์ท้องฟ้ามืดในพื้นที่ส่วนบุคคล": "ที่ดินส่วนบุคคล",
         "อุทยานท้องฟ้ามืด": "อุทยานท้องฟ้ามืด",
         "เขตอนุรักษ์ท้องฟ้ามืดในพื้นที่ชานเมือง": "ชานเมือง",
         "ชุมชนอนุรักษ์ท้องฟ้ามืด": "ชุมชน",
         "อุทยานแห่งชาติ": "อุทยานแห่งชาติ",
         "เขตอนุรักษ์ท้องฟ้ามืดในพื้นที่ส่วนบุคคลเขตชุมชน": "ส่วนบุคคล+ชุมชน"}
holder_site = np.array([SHORT.get(str(t).strip(), str(t).strip()) for t in df["type"].values])
log("ผู้ถือครองจากฟิลด์ type ของทะเบียน: " +
    " · ".join(f"{k}={v}" for k, v in collections.Counter(holder_site).most_common()))

ne = gpd.read_file(os.path.join(CACHE, "ne_10m_admin_0_countries.shp"))
TH = make_valid(ne[ne.NAME == "Thailand"].geometry.iloc[0])
PARTS = list(TH.geoms)
guard_crs(TH.bounds, "thailand-outline", expect="degrees")

def site_context(lat, lon):
    """pop รอบจุดมาจาก hex ที่จุดนั้นตกอยู่ (fallback: hex ที่ใกล้ที่สุด) — ผู้ถือครองมาจากทะเบียนตรง ๆ"""
    c = h3.latlng_to_cell(lat, lon, 5)
    hit = np.where(cell_ids == c)[0]
    if len(hit):
        return float(urban_pop[hit[0]]), "cell"
    d = hav_km(np.array([[lon, lat]]), cen_deg)[0]
    return float(urban_pop[int(np.argmin(d))]), "nearest"

## ตอน 2) อ่านผลวัดจาก Overpass

In [ ]:
# ---------------------------------------------------------------- 2. อ่านผลวัดจาก Overpass
RAW = json.load(open(os.path.join(OSMC, "way_all.json"), encoding="utf-8"))
RADIUS_M = int(RAW["radius_m"])
if len(RAW["sites"]) != 83:
    missing = [i for i in range(83) if str(i) not in RAW["sites"]]
    raise SystemExit(f"ข้อมูลไม่ครบ: มี {len(RAW['sites'])} / 83 จุด (ขาด idx {missing[:8]}...) "
                     f"— รัน tools/collect_day16_osm.py ต่อ (มี resume อัตโนมัติ)")

rows, uid_sets, user_sites = [], [], collections.defaultdict(set)
for i in range(83):
    recs = RAW["sites"][str(i)]
    users = collections.Counter(r[1] for r in recs)
    yrs = sorted({r[2][:4] for r in recs if len(r) > 2 and r[2][:4].isdigit()})
    upop, how = site_context(float(df.latitude.iloc[i]), float(df.longitude.iloc[i]))
    holder = holder_site[i]
    for u in users:
        user_sites[u].add(i)
    uid_sets.append(set(users))
    top_u, top_n = (users.most_common(1)[0] if users else ("", 0))
    rows.append(dict(idx=i, name=df.name.iloc[i], province=str(df.province.iloc[i]),
                     ways=len(recs), users=len(users), top1_user=top_u,
                     top1_share_pct=round(100 * top_n / len(recs), 2) if recs else np.nan,
                     years=f"{yrs[0]}–{yrs[-1]}" if yrs else "",
                     urban_pop_30km=upop, holder=holder, join_how=how))
    if i < 5 or i % 20 == 0:
        log(f"[{i:02d}] {df.name.iloc[i][:24]:24} way {len(recs):>5,} · คน {len(users):>4} · "
            f"อันดับ 1 {top_u[:12]:12} {rows[-1]['top1_share_pct']}% · เมือง {upop:>10,.0f}")

S = pd.DataFrame(rows)
assert len(S) == 83
assert (S.users.values <= np.where(S.ways.values > 0, S.ways.values, 1)).all(), "คนต้องไม่เกิน way"
guard_variance(S.ways.values.astype(float), "ways-per-site", min_spread_km=1.0)

union_users = len(set().union(*uid_sets))
multi_site = {u: s for u, s in user_sites.items() if len(s) >= 2}
top_overall = collections.Counter({u: len(s) for u, s in user_sites.items()})
MULTI_TOP = [{"user": u, "sites": int(n)} for u, n in top_overall.most_common(12)]
log(f"คนแปลกหน้าไม่ซ้ำทั้งประเทศ {union_users:,} · คนที่เขียนมากกว่า 1 จุดทะเบียน "
    f"{len(multi_site)} คน ({100*len(multi_site)/max(union_users,1):.1f}%)")

## ตอน 3) สถิติข้ามจุด + เทียบผู้ถือครองที่ดิน

In [ ]:
# ------------------------------------------------- 3. สถิติข้ามจุด + เทียบผู้ถือครองที่ดิน
rho_u, p_u = spearmanr(S.urban_pop_30km.values, S.users.values)
rho_w, p_w = spearmanr(S.urban_pop_30km.values, S.ways.values)
by_holder = (S.groupby("holder")
             .agg(sites=("idx", "size"), ways_median=("ways", "median"),
                  ways_mean=("ways", "mean"), users_median=("users", "median"))
             .reset_index().sort_values("ways_median", ascending=False))
AGG = dict(sites=int(len(S)), radius_m=RADIUS_M, box_side_m=RADIUS_M * 2,
           ways_total=int(S.ways.sum()), ways_min=int(S.ways.min()),
           ways_median=float(S.ways.median()), ways_mean=float(S.ways.mean()),
           ways_max=int(S.ways.max()),
           users_min=int(S.users.min()), users_median=float(S.users.median()),
           users_mean=float(S.users.mean()), users_max=int(S.users.max()),
           users_total_distinct=union_users, people_multi_site=len(multi_site),
           people_multi_site_share_pct=round(100 * len(multi_site) / max(union_users, 1), 2),
           top1_share_min=float(S.top1_share_pct.min()), top1_share_median=float(S.top1_share_pct.median()),
           top1_share_max=float(S.top1_share_pct.max()),
           sites_with_no_way=int((S.ways == 0).sum()),
           sites_under_50_ways=int((S.ways < 50).sum()),
           sites_under_10_people=int((S.users < 10).sum()),
           spearman_urban_vs_users=round(float(rho_u), 3), spearman_p_users=float(p_u),
           spearman_urban_vs_ways=round(float(rho_w), 3), spearman_p_ways=float(p_w))
log(f"way ต่อจุด: ต่ำสุด {AGG['ways_min']} · มัธยฐาน {AGG['ways_median']:,.0f} · "
    f"สูงสุด {AGG['ways_max']:,} · จุดที่ way<50 = {AGG['sites_under_50_ways']} · "
    f"จุดที่คน<10 = {AGG['sites_under_10_people']}")
log(f"เมืองกับคน rho={rho_u:.3f} (p={p_u:.2g}) · เมืองกับ way rho={rho_w:.3f} (p={p_w:.2g})")
scope = json.load(open(os.path.join(OSMC, "scope_compare.json"), encoding="utf-8")) \
    if os.path.exists(os.path.join(OSMC, "scope_compare.json")) else None

## ตอน 4) ภาพ 1 — แผนที่รอบฟ้ามืดบางแค่ไหน

In [ ]:
# ------------------------------------------------- 4. ภาพ 1 — แผนที่รอบฟ้ามืดบางแค่ไหน
def draw_hero():
    fig = plt.figure(figsize=(14.6, 8.8), dpi=150)
    fig.patch.set_facecolor(BG)
    ax = fig.add_axes([0.025, 0.115, 0.655, 0.75]); ax.set_facecolor(BG)
    ax.add_collection(PolyCollection([list(p.exterior.coords) for p in PARTS],
                                     facecolor="#16223a", edgecolor="#33415e", lw=0.7))
    sc = ax.scatter(df.longitude.values, df.latitude.values,
                    s=20 + np.sqrt(S.users.values.clip(min=0)) * 15,
                    c=np.log10(S.ways.values.clip(min=1)), cmap="plasma",
                    edgecolor=INK, lw=0.5, zorder=5)
    ann = [int(S.sort_values("ways").iloc[-1].idx),
           int(S.sort_values("ways").iloc[0].idx),
           int(S.sort_values("ways").iloc[1].idx)]
    for k, i in enumerate(ann):
        i = int(i)
        r = S.iloc[i]
        ax.annotate(f"{r['name'][:18]}\n{int(r.ways):,} way · {int(r.users)} คน",
                    (df.longitude.values[i], df.latitude.values[i]),
                    xytext=(14, 14 if k == 0 else -30), textcoords="offset points",
                    color=INK, fontsize=8.8, zorder=7)
    ax.set_xlim(97.2, 106.6); ax.set_ylim(5.2, 20.9); ax.set_aspect("equal"); ax.axis("off")
    cb = fig.add_axes([0.688, 0.16, 0.016, 0.40])
    plt.colorbar(sc, cax=cb).set_label(T("log10 จำนวน way"), color=INK, fontsize=9)
    cb.tick_params(colors=INK, labelsize=8.5)
    ax.set_title(T("83 จุดทะเบียนฟ้ามืด · ขนาดวง = จำนวนคน · สี = ปริมาณแผนที่ที่ถูกเขียนรอบจุด"),
                 color=INK, fontsize=12, pad=10)
    txt = fig.add_axes([0.735, 0.10, 0.255, 0.82]); txt.axis("off")
    lines = [(T("way รวมทุกจุด"), f"{AGG['ways_total']:,}"),
             (T("way ต่อจุด · ต่ำสุด"), f"{AGG['ways_min']}"),
             (T("way ต่อจุด · มัธยฐาน"), f"{AGG['ways_median']:,.0f}"),
             (T("way ต่อจุด · สูงสุด"), f"{AGG['ways_max']:,}"),
             (T("จุดที่ way < 50"), f"{AGG['sites_under_50_ways']}"),
             (T("จุดที่ way = 0"), f"{AGG['sites_with_no_way']}"),
             ("", ""),
             (T("คนไม่ซ้ำทั้งประเทศ"), f"{union_users:,}"),
             (T("คนต่อจุด · ต่ำสุด"), f"{AGG['users_min']}"),
             (T("คนต่อจุด · มัธยฐาน"), f"{AGG['users_median']:.0f}"),
             (T("คนต่อจุด · สูงสุด"), f"{AGG['users_max']:,}"),
             (T("จุดที่คน < 10"), f"{AGG['sites_under_10_people']}"),
             (T("คนที่เขียน >1 จุด"), f"{len(multi_site)}"),
             (T("  = ร้อยละของคนทั้งหมด"), f"{AGG['people_multi_site_share_pct']:.1f}%"),
             ("", ""),
             (T("Spearman เมืองกับคน"), f"{rho_u:.3f}"),
             (T("Spearman เมืองกับ way"), f"{rho_w:.3f}")]
    y = 0.98
    for k, v in lines:
        if not k:
            y -= 0.028; continue
        txt.text(0.0, y, k, color=MUTED, fontsize=10.2, va="top")
        txt.text(1.0, y, v, color=ACCENT, fontsize=10.2, va="top", ha="right")
        y -= 0.055
    fig.text(0.025, 0.062, T(f"แผนที่ไทยรอบจุดทะเบียนเป็นผลงานของ {ATTRIBUTION} — "
                             f"คนแปลกหน้า {union_users:,} คน"), color=INK, fontsize=11.5)
    fig.text(0.025, 0.034, T(f"กล่องที่วัดคือ {RADIUS_M*2:,} × {RADIUS_M*2:,} ม. รอบพิกัดแต่ละจุด "
                             "· นับเฉพาะ way (เส้น/พื้นที่) ไม่นับ node"),
             color=MUTED, fontsize=9.8)
    fig.text(0.025, 0.012, T("จุดที่ฟ้าดีที่สุดมักเป็นจุดที่แผนที่บางที่สุด — "
                             "แผนที่หนาแน่นตามคนที่อยู่ ไม่ใช่ตามความมืดของฟ้า"),
             color=MUTED, fontsize=9.8)
    fig.suptitle(T("แผนที่นี้ไม่ได้ผมวาด แต่ทำโดยคนแปลกหน้า — "
                   "และที่ฟ้ามืดที่สุด พวกเขาแทบไม่ได้เขียนอะไรเลย"),
                 color=INK, fontsize=15.5, y=0.972)
    p = os.path.join(OUT, "day16_collaborative.png")
    fig.savefig(p, facecolor=BG); plt.close(fig)
    log(f"เขียน {p}")
    return p

## ตอน 5) ภาพ 2 — เมืองกำหนดว่าใครเขียน

In [ ]:
# ------------------------------------------------- 5. ภาพ 2 — เมืองกำหนดว่าใครเขียน
def draw_who():
    fig, (ax, ax2) = plt.subplots(1, 2, figsize=(14.8, 5.9), dpi=150)
    fig.patch.set_facecolor(BG)
    ax.set_facecolor("#111c2b")
    x = np.clip(S.urban_pop_30km.values, 1, None)
    sc = ax.scatter(x, S.ways.values, s=54, c=S.users.values, cmap="viridis",
                    edgecolor=INK, lw=0.5)
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlabel(T("ประชากรเมืองในรัศมี 30 กม. รอบจุด (log) — จาก Day 9"))
    ax.set_ylabel(T(f"จำนวน way ในกล่อง {RADIUS_M*2:,} ม. รอบจุด (log)"))
    ax.set_title(T(f"แผนที่ถูกเขียนตามคนที่อยู่ ไม่ใช่ตามความมืด\n"
                   f"Spearman rho = {rho_w:.3f} (p = {p_w:.2g})"), color=INK, fontsize=12)
    ax.grid(color="#26364f", lw=0.5, alpha=0.6); ax.tick_params(colors=INK, labelsize=10)
    for s in ax.spines.values(): s.set_color("#33415e")
    cb = fig.add_axes([0.472, 0.17, 0.012, 0.30])
    plt.colorbar(sc, cax=cb).set_label(T("จำนวนคน"), color=INK, fontsize=9)
    cb.tick_params(colors=INK, labelsize=8)

    ax2.set_facecolor("#111c2b")
    bins = np.linspace(0, max(S.ways.max(), 10), 31)
    ax2.hist(S.ways.values, bins=bins, color=TEAL, edgecolor=BG, lw=0.4)
    ax2.axvline(S.ways.median(), color=INK, ls="--", lw=1.2)
    ax2.text(S.ways.median() * 1.05, ax2.get_ylim()[1] * 0.86,
             T(f"มัธยฐาน {S.ways.median():,.0f}"), color=INK, fontsize=10)
    ax2.set_xlabel(T("จำนวน way รอบจุด"))
    ax2.set_ylabel(T("จำนวนจุดทะเบียน"))
    ax2.set_title(T(f"จุดที่แทบไม่มีอะไรในแผนที่เลย\n"
                    f"{AGG['sites_under_50_ways']} จาก 83 จุดมี way น้อยกว่า 50"),
                  color=INK, fontsize=12)
    ax2.grid(color="#26364f", lw=0.5, alpha=0.6); ax2.tick_params(colors=INK, labelsize=10)
    for s in ("top", "right"): ax2.spines[s].set_visible(False)
    for s in ("left", "bottom"): ax2.spines[s].set_color("#33415e")
    fig.suptitle(T("collaborative ที่วัดได้ต้องตอบสองคำถาม: กว้างพอไหม (จำนวนคน) "
                   "และครอบคลุมพอไหม (จำนวนวัตถุ)"), color=INK, fontsize=14.5, y=0.99)
    p = os.path.join(OUT, "day16_who_draws.png")
    fig.savefig(p, facecolor=BG, bbox_inches="tight"); plt.close(fig)
    log(f"เขียน {p}")
    return p

## ตอน 6) ภาพ 3 — ผู้ถือครอง ↔ ใครเขียน + คนที่เขียนหลายที่

In [ ]:
# ------------------------------------------------- 6. ภาพ 3 — ผู้ถือครอง ↔ ใครเขียน + คนที่เขียนหลายที่
def draw_holder():
    fig, (ax, ax2) = plt.subplots(1, 2, figsize=(14.8, 5.9), dpi=150)
    fig.patch.set_facecolor(BG)
    B = by_holder[by_holder.sites >= 3].reset_index(drop=True)
    ax.set_facecolor("#111c2b")
    idx = np.arange(len(B))
    ax.bar(idx - 0.18, B.ways_median.values, width=0.34, color=TEAL, label=T("way · มัธยฐาน"))
    ax.bar(idx + 0.18, B.users_median.values, width=0.34, color=ACCENT, label=T("คน · มัธยฐาน"))
    ax.set_xticks(idx)
    ax.set_xticklabels([f"{h}\n({n} จุด)" for h, n in zip(B.holder.values, B.sites.values)],
                       color=INK, fontsize=9)
    ax.set_yscale("log"); ax.set_ylabel(T("ค่ามัธยฐานต่อจุด (log)"))
    ax.set_title(T("แผนที่รอบฟ้ามืดเขียนไว้ต่างกันตามสภาพการถือครองที่ดิน"),
                 color=INK, fontsize=12)
    ax.legend(facecolor="#111c2b", edgecolor="#33415e", labelcolor=INK, fontsize=9)
    ax.grid(color="#26364f", lw=0.5, alpha=0.6, axis="y"); ax.tick_params(colors=INK, labelsize=9)
    for s in ("top", "right"): ax.spines[s].set_visible(False)
    for s in ("left", "bottom"): ax.spines[s].set_color("#33415e")

    ax2.set_facecolor("#111c2b"); ax2.axis("off")
    ax2.text(0.02, 0.955, T("คนที่เขียนแผนที่ครอบคลุมหลายจุดทะเบียนที่สุด"),
             color=INK, fontsize=12.5)
    ax2.text(0.74, 0.955, T("จำนวนจุด (จาก 83)"), color=MUTED, fontsize=10.5)
    y = 0.88
    for m in MULTI_TOP[:10]:
        ax2.text(0.02, y, m["user"][:32], color="#cfe0f0", fontsize=10.5)
        ax2.text(0.80, y, f"{m['sites']}", color=ACCENT, fontsize=10.5)
        y -= 0.072
    ax2.text(0.02, 0.055, T(f"คนแปลกหน้า {len(multi_site)} จาก {union_users:,} คน "
                            f"({AGG['people_multi_site_share_pct']:.1f}%) เขียนไว้มากกว่าหนึ่งจุด "
                            f"— collaborative ของแผนที่ไทยรอบฟ้ามืดพึ่งคนกลุ่มเล็กที่เดินทางไกล"),
             color=MUTED, fontsize=10, va="top")
    fig.suptitle(T("ใครเป็นคนเขียน? คนกลุ่มเล็กที่เขียนหลายที่ "
                   "กับที่มืดที่สุดที่ไม่มีใครเขียนเลย"), color=INK, fontsize=14.5, y=0.99)
    p = os.path.join(OUT, "day16_holders.png")
    fig.savefig(p, facecolor=BG, bbox_inches="tight"); plt.close(fig)
    log(f"เขียน {p}")
    return p

P1, P2, P3 = draw_hero(), draw_who(), draw_holder()

## ตอน 7) artifact

In [ ]:
# ---------------------------------------------------------------- 7. artifact
S.round(3).to_csv(os.path.join(OUT, "day16_sites_osm.csv"), index=False)
by_holder.round(3).to_csv(os.path.join(OUT, "day16_by_holder.csv"), index=False)
STATS = dict(
    theme="Do a map with a friend, colleague or a stranger. AI doesn't count as a friend :)",
    theme_source="30daymapchallenge.com หัวข้อ Day 16 'Collaborative map' (อ่านสด 07:00 วันที่ 17 พ.ย.)",
    generated_local=time.strftime("%Y-%m-%d %H:%M:%S"),
    posted_late=T("แผ่นนี้วัดและเขียนเช้าวันที่ 17 พ.ย. สำหรับโจทย์วันที่ 16 "
                  "(คืนวันที่ 16 เซิร์ฟเวอร์ Overpass ไม่ตอบ และวิธีวัดด้วย changesets ใช้ไม่ได้)"),
    interpretation=T("ชวนคนไม่ทัน จึงทำแผนที่ที่สร้างจากผลงานของคนแปลกหน้าจริง ๆ "
                     "และวัดว่าพวกเขาคือใคร ครอบคลุมแค่ไหน — ไม่ใช่การสมมติว่า AI เป็นผู้ร่วมทำงาน"),
    provenance=dict(source="OpenStreetMap ผ่าน Overpass API",
                    endpoints=["overpass-api.de", "overpass.kumi.systems"],
                    query_template=f"[out:csv(::id,::user,::timestamp,::lat,::lon,false)]; "
                                   f"way(around:{RADIUS_M}, lat, lon); out center meta;",
                    attribution_definition="จุดศูนย์กลางของ way อยู่ภายใน "
                                           f"{RADIUS_M:,} ม. จากพิกัดจุดทะเบียน",
                    geometry_measured="way เท่านั้น ไม่นับ node",
                    attribution=ATTRIBUTION, collected_by="tools/collect_day16_osm.py",
                    dropped_far_ways=int(RAW.get("dropped_far_ways_total", 0)),
                    collected_window=[RAW.get("fetched_utc"), RAW.get("fetched_utc")]),
    scope_compare=scope,
    rejected_numbers=dict(
        changeset_users_distinct=172, changeset_changes_total=1624027,
        changeset_shared_between_far_sites=44, site_distance_km=1463,
        inside_box_min=0, inside_box_median=0, inside_box_max=39,
        meaning="ตัวเลขจากวิธีวัดด้วย changesets API ที่ *ทิ้งแล้ว* เก็บไว้เพราะโพสต์อ้างถึง "
                "มันไม่ใช่ความจริงเกี่ยวกับสถานที่ แต่เป็นผลของ bbox ของ changeset ที่ครอบคลุมทั้งประเทศ"),
    aggregates=AGG,
    per_site=[{k: (None if (isinstance(v, float) and pd.isna(v)) else v) for k, v in r.items()}
              for r in S.to_dict("records")],
    by_holder=[{k: (None if (isinstance(v, float) and pd.isna(v)) else v) for k, v in r.items()}
               for r in by_holder.to_dict("records")],
    people_covering_many_sites=MULTI_TOP,
    failed_attempts=T("สองทางที่ลองแล้วทิ้งในคืนวันที่ 16: (1) Overpass ด้วย `out;` "
                      "ซึ่งไม่คืนคอลัมน์ user/timestamp ต้องเป็น `out meta;` "
                      "(2) changesets API ซึ่ง bbox ของ changeset ครอบคลุมทั้งประเทศ "
                      "จุดห่าง 1,463 กม. แชร์ changeset กัน 44 จาก 100 ชุด"),
    cross_check_series=dict(register_total=int(len(df)), sites_measured=int(len(S)),
                            day09_field_used="urban_pop_30km",
                            day12_field_used="type→SHORT (ผู้ถือครอง)",
                            day12_private_sites=int((holder_site == "ที่ดินส่วนบุคคล").sum()),
                            hex_join_direct=int((S.join_how == "cell").sum()),
                            hex_join_nearest=int((S.join_how == "nearest").sum()),
                            note="ประชากรเมืองรอบจุดและผู้ถือครองที่ดินอ่านจาก artifact Day 9 / Day 12 ไม่คำนวณใหม่"),
    figures=["day16_collaborative.png", "day16_who_draws.png", "day16_holders.png"],
    metric_definitions={
        "ways": f"จำนวนวัตถุแบบ way (เส้น/พื้นที่) ที่จุดศูนย์กลางอยู่ภายใน {RADIUS_M:,} ม. จากจุดทะเบียน",
        "users": "username ที่ต่างกันของ *การแก้ไขล่าสุด* ของ way เหล่านั้น (หนึ่งคนนับครั้งเดียวต่อจุด)",
        "top1_share_pct": "way ของคนที่แก้มากที่สุดอันดับหนึ่ง ÷ way ทั้งหมด × 100 (ร้อยละ)",
        "years": "ช่วงปีของ timestamp (เวลาแก้ล่าสุดของวัตถุแต่ละชิ้น) ที่เห็นในกล่องนั้น ไม่ใช่ปีแรกที่เริ่มเขียน",
        "users_total_distinct": "username ที่ไม่ซ้ำกันเมื่อนำกล่องของทั้ง 83 จุดมารวมกัน",
        "people_multi_site": "จำนวนคนที่ปรากฏในกล่องของมากกว่าหนึ่งจุดทะเบียน",
        "spearman_urban_vs_ways": f"สหสัมพันธ์อันดับระหว่างประชากรเมืองรอบจุด (Day 9) กับจำนวน way (n = 83 จุด)",
        "spearman_urban_vs_users": "สหสัมพันธ์อันดับระหว่างประชากรเมืองรอบจุดกับจำนวนคนเขียนแผนที่ (n = 83 จุด)",
        "holder": "สภาพการถือครองที่ดินของ *ตัวจุดทะเบียน* เอง จากฟิลด์ type ของทะเบียน (ย่อตาม Day 12) ไม่ใช่ของ hex",
        "dropped_far_ways": "way ที่ Overpass ส่งกลับมาเพราะวัตถุแตะกล่อง แต่จุดศูนย์กลางอยู่นอก 1 กม. จึงถูกตัดออก"},
    note="ตัวเลขทุกตัวอ่านจาก cache/day16/way_all.json ที่ collect_day16_osm.py ดึงไว้ "
         "ค่าเป็น snapshot ของเช้าวันที่ 17 พ.ย. และจะเปลี่ยนเมื่อ OSM ถูกแก้เพิ่ม")
json.dump(STATS, open(os.path.join(OUT, "day16_stats.json"), "w", encoding="utf-8"),
          ensure_ascii=False, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o))
log("เขียน day16_stats.json + day16_sites_osm.csv + day16_by_holder.csv")

## ตอน 8) referee

In [ ]:
# ---------------------------------------------------------------- 8. referee
for f in ["day16_collaborative.png", "day16_who_draws.png", "day16_holders.png",
          "day16_sites_osm.csv", "day16_by_holder.csv", "day16_stats.json"]:
    assert os.path.getsize(os.path.join(OUT, f)) > 0, f"ไฟล์ว่าง: {f}"
json.load(open(os.path.join(OUT, "day16_stats.json"), encoding="utf-8"))
lint_thai(" ".join(TXT) + " " + __doc__, "day16-texts")
assert union_users >= int(S.users.max()), "ผลรวมไม่ซ้ำต้องไม่น้อยกว่าจุดที่มากที่สุด"
assert AGG["sites_under_50_ways"] + int((S.ways >= 50).sum()) == len(S)
assert AGG["ways_total"] == int(S.ways.sum())
_src = os.path.abspath(__file__) if "__file__" in globals() else None
if _src and os.path.exists(_src):
    _bad = []
    for _n, _l in enumerate(open(_src, encoding="utf-8").read().split(chr(10)), 1):
        if any(0x0E00 <= ord(c) <= 0x0E7F for c in _l):
            for c in _l:
                o = ord(c)
                if (0x0100 <= o <= 0x017F) or (0x1E00 <= o <= 0x1EFF) or \
                   (0x3000 <= o <= 0x30FF) or (0xAC00 <= o <= 0xD7AF) or (0x4E00 <= o <= 0x9FFF):
                    _bad.append((_n, c))
    assert not _bad, f"พบอักษรไม่ใช่ไทยในบรรทัดไทย: {_bad[:6]}"
    log("ตรวจ unicode ทั้งไฟล์แล้ว")
log("สรุป: " + json.dumps(dict(sites=len(S), ways_total=AGG["ways_total"],
                               ways_min=AGG["ways_min"], ways_median=AGG["ways_median"],
                               ways_max=AGG["ways_max"], sites_under_50=AGG["sites_under_50_ways"],
                               union_users=union_users, users_median=AGG["users_median"],
                               multi_site=len(multi_site), rho_ways=round(float(rho_w), 3)),
                          ensure_ascii=False))
print("DONE day16 ใน %.1f วิ" % (time.time() - T0))
_LOGF.close()